# Maxwell's demon, the Szilard engine and Landauer's principle

*thermodynamics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/maxwell_demon)

A demon looks at which half of a box a Brownian particle is in and slides a partition to let it push, extracting work from a single heat bath. Batches of overdamped Langevin engines show the bill: the work is at most $kT\,I \le kT\ln 2$ per bit, with $I$ the mutual information of a noisy measurement, erasing the bit costs at least $kT\ln 2$ (Landauer), and the Sagawa–Ueda equality $\langle e^{-\beta(W-\Delta F)-I}\rangle = 1$ restores the second law once information is counted. Played backwards, the log ratio of path probabilities is the entropy produced.

**Vector form**

$$\gamma\,\dot x = -\partial_x V(x;p(t)) + \sqrt{2\gamma kT}\,\xi(t),\qquad W = \int \partial_p V\,\dot p\,dt,\qquad \langle W_{ext}\rangle \le kT\,I,\quad I = \ln 2 + \varepsilon\ln\varepsilon + (1-\varepsilon)\ln(1-\varepsilon),\qquad \big\langle e^{-\beta(W-\Delta F) - I}\big\rangle = 1,\qquad W_{erase} \ge kT\ln 2,\qquad \frac{P_F(W)}{P_R(-W)} = e^{\beta(W-\Delta F)}$$

**Index (tensor) form**

$$y_b = x_b + F_b\,\Delta t + \sqrt{2\Delta t}\,\xi_b,\qquad \ln r_b = V_b(x) - V_b(y) + \ln\frac{q(x_b|y_b)}{q(y_b|x_b)},\qquad W_b \mathrel{+}= V(x_b;p_{b,k+1}) - V(x_b;p_{b,k}),\qquad I_b = \ln\frac{P(m_b|s_b)}{P(m_b)},\qquad \frac{1}{B}\textstyle\sum_b e^{-(W_b - \Delta F_{m_b}) - I_b} \to 1$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V(x;p,s)$ | `potential(x, p, side)` | (B,) | soft box walls at $0$ and $2a$ plus the partition at $p$, felt from the side $s=\pm1$ the particle is on |
| $x_b,\ s_b$ | `sample_equilibrium(B, a, side, ...)` | (B,) | every engine starts in equilibrium $e^{-V/kT}/Z$ with the partition in the middle |
| $m_b$ | `where(wrong, -side, side)` | (B,) | the demon's reading, wrong with probability $\varepsilon$ |
| $p_b(t)$ | `ramp(t, p0, p1, tau)` | (B,) | feedback: the partition slides to $2a(1-\varepsilon)$ on the side the demon believes in |
| $y_b,\ r_b$ | `mala_step(x, V, F, p, side, dt, ...)` | (B,) | Euler–Maruyama proposal for all engines at once, accepted with Metropolis–Hastings so $e^{-V/kT}$ is exactly stationary |
| $W_b$ | `W += potential(x, p_new) - potential(x, p)` | (B,) | work done on the particle when the wall moves at fixed $x$ |
| $I_b$ | `trajectory_information(correct, eps)` | (B,) | $\ln 2(1-\varepsilon)$ if the demon was right, $\ln 2\varepsilon$ if wrong; its average is $I$ |
| $Z(p,s)$ | `partition_function(p, side, grid)` | (P,) | trapezoid rule over a grid for a batch of partition positions: $\Delta F = -kT\ln Z_1/Z_0$ |
| $\sigma_b$ | `log_path + ln_rho0 - ln_rhoN` | (B,) | $\ln P_F[x]/P_R[\tilde x]$ from the transition probabilities of every step plus the boundary terms |

Units: $kT = 1$, mobility $1$ (so $D = 1$), half-box width $a = 1$, time in $a^2/D$. Maxwell (1867) imagined a being that sorts molecules and beats the second law; Szilard (1929) reduced it to one particle and one bit, worth at most $kT\ln 2$ of work. Landauer (1961) and Bennett (1982) settled the paradox: the demon must eventually erase its memory, which costs at least $kT\ln 2$ per bit, so the cycle as a whole obeys the second law. Sagawa and Ueda (2010) made this quantitative for feedback with errors: $\langle e^{-\beta(W-\Delta F)-I}\rangle = 1$, so $\langle W_{ext}\rangle \le kT\langle I\rangle - \langle\Delta F\rangle$. These are proven theorems for this kind of stochastic dynamics, and the simulation checks them numerically. Model: the walls are stiff springs $\kappa u^2/2$ ($\kappa = 500\,kT/a^2$), so free energies are computed numerically from $Z$, not set to their hard-wall values. The partition is impenetrable, so each particle keeps its side $s$. The Langevin steps are Metropolis-corrected (MALA), so the Boltzmann distribution, Jarzynski, Sagawa–Ueda and Crooks hold exactly up to sampling noise. The erasure starts from the partitioned box and ends with the piston at the middle, which makes $\Delta F = kT\ln 2$ exact even with soft walls. The ledger counts the work of moving the partition. Pulling the partition out at the end returns its small spring energy. Real data, as cited numbers only (no figures reproduced): Bérut et al., Nature 483, 187 (2012) and the detailed version in J. Stat. Mech. P06015 (2015) erased a bit stored in a $2\,\mu$m silica bead in an optical double well. The mean heat fell with the cycle time $\tau$ as $\langle Q\rangle = kT\ln 2 + B/\tau$ with $B = 8.15\,kT\cdot$s, and a two-parameter fit gave an asymptote $A = 0.72\,kT$ ($\pm 0.15\,kT$), close to $kT\ln 2 \approx 0.693\,kT$. The simulated erasure gives the same $1/\tau$ law in its own units. Time reversal: the last part of the movie plays the erasure backwards. For each trajectory, the log ratio of its forward probability to that of the reversed path under the reversed protocol equals $\beta(W - \Delta F)$, the entropy produced (Crooks 1999). Its average is positive, and $\langle e^{-\sigma}\rangle = 1$. Try: a larger error rate $\varepsilon$ (less information, less work; $\varepsilon = 0.5$ gives nothing), or a shorter $\tau$ (more dissipation).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/maxwell_demon/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    gen = torch.Generator().manual_seed(params["seed"])
    n, eps_sel, tau = params["n_engines"], params["eps"], params["tau"]
    ln2 = math.log(2.0)

    # 1. Szilard engines with a demon, for a batch of measurement error rates at once
    eps = torch.tensor(sorted({0.0, 0.05, 0.1, 0.2, 0.3, 0.5, round(eps_sel, 3)}), dtype=tensors.DTYPE)
    n_steps = math.ceil(tau / tensors.DT - 1e-9)
    rec_a = frame_indices(n_steps + 1, 36)
    sz = solve.szilard(n, eps, tau, gen, dev, record=rec_a)
    E = len(eps)
    W_ext = -sz["W_on"].view(E, n)
    info = sz["info"].view(E, n)
    I_th = equation.mutual_information(eps)
    su = torch.exp(-(sz["W_on"] - sz["dF_m"]) - sz["info"]).view(E, n).mean(1)
    no_info = torch.exp(-(sz["W_on"] - sz["dF_m"])).view(E, n).mean(1)
    print(f"Szilard engines: {n} per error rate, partition moved in τ = {tau} a²/D (kT = 1)")
    print("   ε    I = ln2−H(ε)   ⟨I⟩ sampled   ⟨W_ext⟩ ± se     ⟨e^{-β(W−ΔF)−I}⟩   ⟨e^{-β(W−ΔF)}⟩")
    for e in range(E):
        se = W_ext[e].std() / math.sqrt(n)
        print(f"  {eps[e]:.2f}   {I_th[e]:9.4f}     {info[e].mean():9.4f}    {W_ext[e].mean():7.4f} ± {se:.4f}"
              f"      {su[e]:8.4f}          {no_info[e]:8.4f}")  # fmt: skip
    print(f"Szilard limit kT ln 2 = {ln2:.4f}: every ⟨W_ext⟩ stays below kT·I (second law with information)")
    print(f"removing the partition returns ⟨{-sz['W_remove'].mean():.4f}⟩ kT of spring energy (soft walls)")

    # 2. Landauer erasure, a batch of protocol times
    taus = torch.tensor([0.25, 0.5, 1.0, 2.0, params["tau_erase"]], dtype=tensors.DTYPE).unique()
    rec_b = frame_indices(math.ceil(taus.max().item() / tensors.DT - 1e-9) + 1, 18)
    er = solve.erasure(max(200, n // 2), taus, gen, dev, record=rec_b)
    Wer = er["W"].view(len(taus), -1)
    mean_er, se_er = Wer.mean(1), Wer.std(1) / math.sqrt(Wer.shape[1])
    B_fit = (((mean_er - ln2) / taus).sum() / (1 / taus**2).sum()).item()  # least squares ⟨W⟩ = ln2 + B/τ
    print(f"Landauer erasure (ΔF = {er['dF']:.6f} = kT ln 2): fit ⟨W⟩ = kT ln 2 + B/τ with B = {B_fit:.3f} kT·a²/D")
    for t, m, s, ok in zip(taus, mean_er, se_er, er["success"]):
        print(f"  τ = {t:5.2f}: ⟨W⟩ = {m:.4f} ± {s:.4f} kT   (bit reset in {100 * ok:.1f}% of runs)")
    print("Bérut et al. (2012; 2015): ⟨Q⟩ = kT ln 2 + B/τ with B = 8.15 kT·s; free fit gives A = 0.72 kT (±0.15 kT)")

    # 3. Time reversal: Crooks and path probabilities
    cr = solve.crooks(20000, 0.1, gen, dev)
    slope, intercept = equation.crooks_line(cr["W_forward"], -cr["W_reverse"])
    sig_err = (cr["sigma_path"] - (cr["W_forward"] - cr["dF"])).abs().max().item()
    print(f"Crooks (piston 2a → a in τ = 0.1): ln P_F(W)/P_R(−W) = {slope:.3f} W − {-intercept:.3f};"
          f" ΔF = {cr['dF']:.4f}")  # fmt: skip
    print(f"path log ratio ln P_F[x]/P_R[x̃] = β(W − ΔF) on every trajectory: max error {sig_err:.1e};"
          f" ⟨e^(−σ)⟩ = {torch.exp(-cr['sigma_path']).mean():.4f}, ⟨σ⟩ = {cr['sigma_path'].mean():.3f}")  # fmt: skip

    sel = int((eps - eps_sel).abs().argmin())
    summary = {"eps": eps, "I": I_th, "W_ext": W_ext, "su": su, "sel": sel, "taus": taus, "mean_er": mean_er,
               "se_er": se_er, "B_fit": B_fit, "crooks": cr, "slope": slope, "dt": tensors.DT}  # fmt: skip
    return plot.animate(sz, er, summary, out_dir / "maxwell_demon.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/maxwell_demon/equation.py
"""Physics → tensors: the potential, overdamped Langevin with a Metropolis correction, work, information.

Every function acts elementwise on a batch $b$ of independent particles (no sums over particles except the averages
the averages). Shapes in comments; B = batch.
"""

import math

import torch

from . import tensors


def wall(u: torch.Tensor) -> torch.Tensor:
    """Soft wall energy κ/2 · relu(u)² for a penetration u (any shape)."""
    return 0.5 * tensors.KAPPA * torch.relu(u) ** 2


def potential(x, p, side, tilt: float = 0.0):
    """V(x; p, s) = box walls + partition at p felt from side s (−1 left, +1 right, 0 = no partition) + tilt·x.

    x, p, side: (B,) or broadcastable. A particle on the left (s = −1) is pushed back when x > p;
    on the right when x < p.
    """
    return wall(-x) + wall(x - 2 * tensors.A) + wall(side * (p - x)) + tilt * x


def force(x, p, side, tilt: float = 0.0):
    """F = −∂V/∂x (B,)."""
    k = tensors.KAPPA
    return k * torch.relu(-x) - k * torch.relu(x - 2 * tensors.A) + k * side * torch.relu(side * (p - x)) - tilt


def log_kernel(y, x, f_x, dt: float):
    """log of the Euler–Maruyama (Onsager–Machlup) transition density x → y, without the constant:
    −(y−x−F dt)²/4D dt."""
    return -((y - x - f_x * dt) ** 2) / (4 * dt)


def mala_step(x, v_x, f_x, p, side, dt, noise, uniform, tilt=0.0):
    """One overdamped Langevin step x → x + F dt + √(2 dt) ξ, accepted with the Metropolis–Hastings probability so that
    e^{−V/kT} is exactly stationary. Returns the new x, V, F and the path log ratio ln P[x→x'] − ln P[x'→x] (B,)."""
    y = x + f_x * dt + math.sqrt(2 * dt) * noise
    v_y, f_y = potential(y, p, side, tilt), force(y, p, side, tilt)
    log_q_f, log_q_b = log_kernel(y, x, f_x, dt), log_kernel(x, y, f_y, dt)  # proposal densities x→y and y→x
    log_r = (v_x - v_y) + log_q_b - log_q_f  # Metropolis–Hastings log ratio
    accept = torch.log(uniform) < log_r
    # path weight of this step forward over reversed: P[x→y]/P[y→x] = q(y|x) min(1, r) / q(x|y) min(1, 1/r)
    log_path = log_q_f + log_r.clamp(max=0) - log_q_b - (-log_r).clamp(max=0)
    x_new = torch.where(accept, y, x)
    return x_new, torch.where(accept, v_y, v_x), torch.where(accept, f_y, f_x), torch.where(accept, log_path, 0 * x)


def partition_function(p, side, grid, tilt: float = 0.0):
    """Z(p, s) = ∫ e^{−V(x; p, s)} dx by the trapezoid rule; p, side: (P,), grid (X,) → (P,)."""
    v = potential(grid[None, :], p[:, None], side[:, None], tilt)  # (P, X)
    return torch.trapezoid(torch.exp(-v), grid, dim=1)


def mutual_information(eps):
    """I = ln 2 + ε ln ε + (1−ε) ln(1−ε): what a measurement with error rate ε tells about a fair bit (nats)."""
    xlogx = lambda q: torch.where(q > 0, q * torch.log(q.clamp_min(1e-300)), 0 * q)  # noqa: E731
    return math.log(2.0) + xlogx(eps) + xlogx(1 - eps)


def trajectory_information(correct, eps):
    """Stochastic mutual information I = ln P(m|s)/P(m) for each run: ln 2(1−ε) if the demon was right,
    else ln 2ε (B,)."""
    return torch.where(correct, torch.log(2 * (1 - eps)), torch.log((2 * eps).clamp_min(1e-300)))


def crooks_line(w_f, minus_w_r, bins: int = 24, min_count: int = 40):
    """Least-squares line through ln P_F(W) − ln P_R(−W) on the bins both histograms populate → (slope, intercept)."""
    lo, hi = max(w_f.min(), minus_w_r.min()).item(), min(w_f.max(), minus_w_r.max()).item()
    edges = torch.linspace(lo, hi, bins + 1, dtype=w_f.dtype)
    h_f = torch.histc(w_f, bins, lo, hi) / len(w_f)
    h_r = torch.histc(minus_w_r, bins, lo, hi) / len(minus_w_r)
    keep = (h_f * len(w_f) >= min_count) & (h_r * len(minus_w_r) >= min_count)
    centres = 0.5 * (edges[1:] + edges[:-1])[keep]
    y = torch.log(h_f[keep] / h_r[keep])
    design = torch.stack([centres, torch.ones_like(centres)], 1)
    coef = torch.linalg.lstsq(design, y[:, None]).solution[:, 0]
    return coef[0].item(), coef[1].item()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/maxwell_demon/tensors.py
"""Concrete tensors: units, the box, the soft walls, and equilibrium samples.

Units: kT = 1, mobility = 1 (so D = 1) and the half-box width a = 1. The box is [0, 2a];
the demon's partition sits at x = p; every wall is a stiff spring κu²/2 on the penetration u > 0.
"""

import torch

A = 1.0  # half-box width a: one bit = "left half" or "right half"
KAPPA = 500.0  # wall stiffness (kT / a²): walls are soft over √(π/2κ) ≈ 0.06 a
DT = 8e-4  # time step (a² / D); κ·dt = 0.4, corrected exactly by a Metropolis test (MALA)
DTYPE = torch.float64


def x_grid(device: torch.device, n: int = 8001) -> torch.Tensor:
    """(X,) quadrature grid that covers the box and the soft walls."""
    return torch.linspace(-0.4, 2 * A + 0.4, n, dtype=DTYPE, device=device)


def full(n: int, value: float, device: torch.device) -> torch.Tensor:
    return torch.full((n,), float(value), dtype=DTYPE, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/maxwell_demon/solve.py
"""Solvers: equilibrium sampling, driven Langevin runs, the Szilard engine with a demon, Landauer erasure, Crooks."""

import math

import torch

from problems._common.progress import steps

from . import equation, tensors


def sample_equilibrium(n: int, p: float, side: float, gen, device, tilt: float = 0.0) -> torch.Tensor:
    """n positions drawn from e^{-V(x; p, s)}/Z by inverting the cumulative distribution on the grid (n,)."""
    grid = tensors.x_grid(device)
    rho = torch.exp(-equation.potential(grid, torch.tensor(p), torch.tensor(float(side)), tilt))
    cdf = torch.cumsum(rho, 0)
    cdf = cdf / cdf[-1]
    u = torch.rand(n, generator=gen, dtype=tensors.DTYPE).to(device)
    i = torch.searchsorted(cdf, u).clamp(1, len(grid) - 1)
    w = (u - cdf[i - 1]) / (cdf[i] - cdf[i - 1]).clamp_min(1e-300)
    return grid[i - 1] + w * (grid[i] - grid[i - 1])


def ramp(t: float, p0, p1, tau):
    """Linear protocol p(t) = p0 + (p1 - p0) min(t/tau, 1), per particle (B,)."""
    return p0 + (p1 - p0) * torch.clamp(t / tau, max=1.0)


def run_protocol(x, side, p0, p1, tau, t_total, gen, tilt=0.0, record=()):
    """Drive the partition (or piston) from p0 to p1 in time tau (all (B,)) and integrate overdamped Langevin.

    Work is the energy change at fixed x when p moves: W += V(x, p_{k+1}) - V(x, p_k). The path log ratio sums
    ln P[x_k -> x_{k+1}] - ln P[x_{k+1} -> x_k] over the steps (Euler proposal + Metropolis).
    record: step numbers to store.
    """
    dt = tensors.DT
    n_steps = max(1, math.ceil(t_total / dt - 1e-9))  # the protocol always reaches its end
    p = p0.clone()
    v = equation.potential(x, p, side, tilt)
    W = torch.zeros_like(x)
    log_path = torch.zeros_like(x)
    frames = {}
    record = set(record)
    if 0 in record:
        frames[0] = (x.clone(), p.clone(), W.clone())
    for k in steps(n_steps, "Langevin steps"):
        p_new = ramp((k + 1) * dt, p0, p1, tau)
        v_new = equation.potential(x, p_new, side, tilt)
        W += v_new - v  # work done on the particle by moving the wall
        f = equation.force(x, p_new, side, tilt)
        noise = torch.randn(x.shape, generator=gen, dtype=x.dtype).to(x.device)
        uni = torch.rand(x.shape, generator=gen, dtype=x.dtype).to(x.device)
        x, v, f, lr = equation.mala_step(x, v_new, f, p_new, side, dt, noise, uni, tilt)
        log_path += lr
        p = p_new
        if k + 1 in record:
            frames[k + 1] = (x.clone(), p.clone(), W.clone())
    return {"x": x, "p": p, "W": W, "log_path": log_path, "frames": frames, "n_steps": n_steps}


def partition_z(p, side, device) -> torch.Tensor:
    """Z(p, s) for one side or a batch of partition positions (P,)."""
    grid = tensors.x_grid(device)
    p = torch.as_tensor(p, dtype=tensors.DTYPE, device=device).reshape(-1)
    side = torch.as_tensor(side, dtype=tensors.DTYPE, device=device).reshape(-1).expand_as(p)
    return equation.partition_function(p, side, grid)


def _random_side(n: int, gen, device) -> torch.Tensor:
    return torch.where(torch.rand(n, generator=gen, dtype=tensors.DTYPE) < 0.5, -1.0, 1.0).to(device, tensors.DTYPE)


def szilard(n_per: int, eps, tau: float, gen, device, t_total=None, record=()):
    """Szilard engine with a noisy demon for every error rate eps_e at once (E groups of n_per engines).

    1. Equilibrium with the partition at x = a; the particle's true side s = -1 (left) or +1 (right), prob. 1/2.
    2. The demon reads m = s, wrong with probability eps.
    3. Feedback: the partition slides so that the side the demon believes in gets 2a(1 - eps) of the box (optimal).
    4. The partition is removed (its residual spring energy is returned, W_remove).
    """
    a = tensors.A
    eps = eps.to(device)
    E = len(eps)
    eps_b = eps.repeat_interleave(n_per)  # (B,)
    B = E * n_per
    side = _random_side(B, gen, device)
    x = torch.where(side < 0, sample_equilibrium(B, a, -1.0, gen, device), sample_equilibrium(B, a, 1.0, gen, device))
    wrong = torch.rand(B, generator=gen, dtype=tensors.DTYPE).to(device) < eps_b
    m = torch.where(wrong, -side, side)  # the demon's reading
    p0 = torch.full((B,), a, dtype=tensors.DTYPE, device=device)
    p1 = torch.where(m < 0, 2 * a * (1 - eps_b), 2 * a * eps_b)  # final partition position
    run = run_protocol(x, side, p0, p1, torch.full_like(p0, tau), t_total or tau, gen, record=record)
    W_remove = -equation.wall(side * (run["p"] - run["x"]))  # pulling the partition out returns its spring energy
    z0 = partition_z(a, -1.0, device) + partition_z(a, 1.0, device)  # Z of the partitioned box at the start
    p_fin = torch.cat([2 * a * (1 - eps), 2 * a * eps])  # (2E,) final positions for m = L, R
    z1 = (partition_z(p_fin, -1.0, device) + partition_z(p_fin, 1.0, device)).view(2, E)
    dF_groups = -torch.log(z1 / z0)  # (2, E) Delta F_m = -ln Z(p_f(m)) / Z(a)
    group = torch.arange(E, device=device).repeat_interleave(n_per)
    dF_m = torch.where(m < 0, dF_groups[0, group], dF_groups[1, group])
    info = equation.trajectory_information(~wrong, eps_b)
    return {"W_on": run["W"], "W_remove": W_remove, "dF_m": dF_m, "info": info, "side": side, "m": m,
            "wrong": wrong, "p1": p1, "frames": run["frames"], "eps": eps, "n_per": n_per}  # fmt: skip


def erasure(n_per: int, taus, gen, device, record=()):
    """Landauer erasure of a bit stored as the side of a partitioned box, for every protocol time tau_t at once.

    Start: equilibrium with the partition, side unknown (1 bit). Remove the partition, push a piston from x = 2a to a
    in time tau: the particle ends on the left (bit reset to 0). Delta F = ln[Z(left)+Z(right)] - ln Z(left) = ln 2.
    """
    a = tensors.A
    taus = taus.to(device)
    B = len(taus) * n_per
    tau_b = taus.repeat_interleave(n_per)
    side = _random_side(B, gen, device)
    x = torch.where(side < 0, sample_equilibrium(B, a, -1.0, gen, device), sample_equilibrium(B, a, 1.0, gen, device))
    W_remove = -equation.wall(side * (a - x))  # remove the partition
    piston = torch.full((B,), -1.0, dtype=tensors.DTYPE, device=device)  # the piston pushes from the right
    p0 = torch.full((B,), 2 * a, dtype=tensors.DTYPE, device=device)
    p1 = torch.full((B,), a, dtype=tensors.DTYPE, device=device)
    run = run_protocol(x, piston, p0, p1, tau_b, taus.max().item(), gen, record=record)
    z_left, z_right = partition_z(a, -1.0, device), partition_z(a, 1.0, device)
    dF = (torch.log(z_left + z_right) - torch.log(z_left)).item()
    success = (run["x"] < a + 4 / tensors.KAPPA**0.5).to(tensors.DTYPE).view(len(taus), -1).mean(1)
    return {"W": W_remove + run["W"], "dF": dF, "success": success, "side": side, "taus": taus,
            "frames": run["frames"], "n_per": n_per}  # fmt: skip


def crooks(n: int, tau: float, gen, device, record=()):
    """Forward: compress the empty box with a piston 2a -> a in time tau. Reverse: expand a -> 2a from equilibrium.

    Also returns sigma = ln P_F[x]/P_R[x~]: the log ratio of the transition probabilities along the path plus
    ln rho_eq(x_0) - ln rho_eq(x_N).
    """
    a = tensors.A
    s = torch.full((n,), -1.0, dtype=tensors.DTYPE, device=device)
    big, small = (torch.full((n,), v, dtype=tensors.DTYPE, device=device) for v in (2 * a, a))
    t = torch.full((n,), tau, dtype=tensors.DTYPE, device=device)
    z_big, z_small = partition_z(2 * a, -1.0, device), partition_z(a, -1.0, device)
    dF = (torch.log(z_big) - torch.log(z_small)).item()

    x0 = sample_equilibrium(n, 2 * a, -1.0, gen, device)
    fwd = run_protocol(x0, s, big, small, t, tau, gen, record=record)
    ln_rho0 = -equation.potential(x0, big, s) - torch.log(z_big)
    ln_rho_n = -equation.potential(fwd["x"], small, s) - torch.log(z_small)
    sigma = fwd["log_path"] + ln_rho0 - ln_rho_n

    x0r = sample_equilibrium(n, a, -1.0, gen, device)
    rev = run_protocol(x0r, s, small, big, t, tau, gen, record=record)
    return {"W_forward": fwd["W"], "W_reverse": rev["W"], "dF": dF, "sigma_path": sigma,
            "frames_forward": fwd["frames"], "frames_reverse": rev["frames"]}  # fmt: skip


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/maxwell_demon/plot.py
"""Animate the demon's engines, the work/information ledger, the erasure cost and the time-reversed run."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, save_gif

LANES = 10
LN2 = math.log(2.0)


def animate(sz: dict, er: dict, s: dict, path: Path) -> Path:
    eps, info, W_ext, sel = s["eps"].numpy(), s["I"].numpy(), s["W_ext"].numpy(), s["sel"]
    n = sz["n_per"]
    lanes = np.arange(sel * n, sel * n + LANES)  # the first engines of the selected error rate
    keys_a = sorted(sz["frames"])
    xa = np.stack([sz["frames"][k][0].numpy() for k in keys_a])  # (Fa, B)
    pa = np.stack([sz["frames"][k][1].numpy() for k in keys_a])
    wa = -np.stack([sz["frames"][k][2].numpy() for k in keys_a]).reshape(len(keys_a), len(eps), n).mean(2)  # (Fa, E)
    ta = np.array(keys_a) * s["dt"]
    wrong = sz["wrong"].numpy()

    keys_b = sorted(er["frames"])
    m_er = er["n_per"]
    taus = s["taus"].numpy()
    lanes_b = np.arange((len(taus) - 1) * m_er, (len(taus) - 1) * m_er + LANES)  # slowest erasure group
    xb = np.stack([er["frames"][k][0].numpy()[lanes_b] for k in keys_b])
    pb = np.stack([er["frames"][k][1].numpy()[lanes_b] for k in keys_b])
    tb = np.array(keys_b) * s["dt"]
    Fa, Fb = len(keys_a), len(keys_b)
    total = Fa + 2 * Fb

    fig = plt.figure(figsize=(11.5, 6.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1, 1])
    ax = fig.add_subplot(gs[:, 0])
    ax_l, ax_w = fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])
    ax_e, ax_c = fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[1, 2])

    # --- engines: one box per lane ---
    for j in range(LANES):
        ax.add_patch(plt.Rectangle((0, j - 0.38), 2, 0.76, fill=False, ec="#374151", lw=1.2))
    ax.axvline(1.0, color="#9ca3af", ls="--", lw=0.8)
    bars = [ax.plot([], [], "-", color="#111827", lw=3)[0] for _ in range(LANES)]
    dots = ax.scatter(np.zeros(LANES), np.arange(LANES), s=40, zorder=3)
    ax.set_xlim(-0.1, 2.1)
    ax.set_ylim(-0.7, LANES - 0.3)
    ax.set_yticks([])
    ax.set_xlabel("position $x/a$")
    title = ax.set_title("", fontsize=9)

    # --- ledger for the selected ε ---
    ax_l.axhline(info[sel], color=ACCENT[2], ls="--", lw=1.2, label=f"$kT\\,I$ = {info[sel]:.3f} (info gained)")
    ax_l.axhline(LN2, color="#9ca3af", ls=":", lw=1.2, label="$kT\\ln 2$ (Szilard)")
    led = ax_l.plot([], [], color=ACCENT[0], lw=2, label="$\\langle W_{ext}\\rangle$")[0]
    ax_l.set_xlim(0, ta[-1])
    ax_l.set_ylim(min(-0.05, wa[:, sel].min() - 0.05), 0.8)
    ax_l.set_xlabel("time $t$ ($a^2/D$)")
    ax_l.set_ylabel("work out / kT")
    ax_l.legend(fontsize=7, loc="lower right")
    ax_l.set_title(f"ledger, demon error ε = {eps[sel]:.2f}", fontsize=9)

    # --- W_ext vs I for every ε ---
    grid = np.linspace(0, LN2, 50)
    ax_w.plot(grid, grid, color=ACCENT[2], ls="--", lw=1.2, label="bound $W_{ext} = kT\\,I$")
    ax_w.axhline(LN2, color="#9ca3af", ls=":", lw=1)
    pts = ax_w.scatter(info, wa[0], c=ACCENT[0], s=28, zorder=3, label="engines")
    for e in range(len(eps)):
        ax_w.annotate(
            f"ε={eps[e]:.2f}", (info[e], W_ext[e].mean()), fontsize=6, xytext=(3, -9), textcoords="offset points"
        )
    ax_w.set_xlim(-0.03, 0.75)
    ax_w.set_ylim(-0.1, 0.78)
    ax_w.set_xlabel("mutual information $I$ (nats)")
    ax_w.set_ylabel("$\\langle W_{ext}\\rangle$ / kT")
    ax_w.legend(fontsize=7, loc="upper left")
    su = s["su"].numpy()
    ax_w.set_title(
        f"Sagawa–Ueda $\\langle e^{{-\\beta(W-\\Delta F)-I}}\\rangle$ = {su.min():.2f}…{su.max():.2f}", fontsize=9
    )

    # --- erasure cost ---
    tt = np.geomspace(taus.min() * 0.8, taus.max() * 1.3, 100)
    ax_e.plot(tt, LN2 + s["B_fit"] / tt, color=ACCENT[3], lw=1, label=f"fit $kT\\ln2 + B/\\tau$, B={s['B_fit']:.2f}")
    ax_e.axhline(LN2, color="#9ca3af", ls=":", lw=1.2, label="Landauer $kT\\ln 2$")
    m_er, se_er = s["mean_er"].numpy(), s["se_er"].numpy()
    ax_e.errorbar(taus, m_er, yerr=2 * se_er, fmt="o", color=ACCENT[1], ms=5, label="erasure ⟨W⟩ ± 2 se")
    ax_e.set_xscale("log")
    ax_e.set_xticks(taus, [f"{t:g}" for t in taus])
    ax_e.xaxis.set_minor_formatter(plt.NullFormatter())
    ax_e.set_xlabel("erasure time $\\tau$ ($a^2/D$)")
    ax_e.set_ylabel("work in / kT")
    ax_e.set_ylim(0, max(m_er) + 0.4)
    ax_e.legend(fontsize=7, loc="upper right")
    ax_e.set_title("Landauer: erasing 1 bit (Bérut 2012: → kT ln 2)", fontsize=9)

    # --- Crooks ---
    cr = s["crooks"]
    wf, wr = cr["W_forward"].numpy(), -cr["W_reverse"].numpy()
    bins = np.linspace(min(wf.min(), wr.min()), 4.0, 45)
    ax_c.hist(wf, bins, density=True, alpha=0.6, color=ACCENT[0], label="$P_F(W)$ compress")
    ax_c.hist(wr, bins, density=True, alpha=0.6, color=ACCENT[4], label="$P_R(-W)$ expand")
    ax_c.axvline(cr["dF"], color="#111827", lw=1.2, label=f"$\\Delta F$ = {cr['dF']:.3f}")
    ax_c.set_yscale("log")
    ax_c.set_xlabel("work $W$ / kT")
    ax_c.legend(fontsize=7)
    ax_c.set_title(f"Crooks: curves cross at ΔF, slope {s['slope']:.2f}", fontsize=9)
    for a_ in (ax_w, ax_e, ax_c):
        a_.set_visible(False)

    def update(f: int):
        if f < Fa:  # demon feedback
            x, p = xa[f, lanes], pa[f, lanes]
            col = np.where(wrong[lanes], "#dc2626", ACCENT[0])
            title.set_text(f"Szilard engines, demon error ε = {eps[sel]:.2f}\nblue: read correctly, red: misread"
                           f"   t = {ta[f]:.2f}")  # fmt: skip
            led.set_data(ta[: f + 1], wa[: f + 1, sel])
            pts.set_offsets(np.c_[info, wa[f]])
            ax_w.set_visible(f > Fa // 3)
        else:  # erasure, forward then backward in time
            g = f - Fa
            k = g if g < Fb else 2 * Fb - 1 - g
            x, p = xb[k], pb[k]
            col = np.full(LANES, "#6b7280")
            arrow = "forward" if g < Fb else "played BACKWARDS"
            title.set_text(f"erasing the bit, τ = {taus[-1]:.1f}: piston 2a → a ({arrow})\n"
                           f"t = {tb[k]:.2f}; reversed: σ = ln P_F/P_R = β(W−ΔF)")  # fmt: skip
            ax_w.set_visible(True)
            ax_e.set_visible(True)
            ax_c.set_visible(g >= Fb)
        dots.set_offsets(np.c_[x, np.arange(LANES)])
        dots.set_color(col)
        for j in range(LANES):
            bars[j].set_data([p[j], p[j]], [j - 0.38, j + 0.38])

    fig.tight_layout(h_pad=1.6)
    return save_gif(fig, update, list(range(total)), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'eps': 0.1,
    'tau': 2.5,
    'tau_erase': 4.0,
    'n_engines': 1500,
    'seed': 1
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.maxwell_demon")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.maxwell_demon.main")
problem_dir = Path("problems/thermodynamics/maxwell_demon")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
